In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colors as mcolors
import numpy as np
import math
import random
import sympy
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [2]:
def generator_gap(x):
    return (-1/x)*math.log(random.uniform(0,1))

In [3]:
lamd=10
mu_1=2
mu_2=8
kanal_mu={
    "канал 1": mu_1,
    "канал 2": mu_2
}
N=1000


In [4]:
class RequestCar:
    def __init__(self, id, input_time):
        self.id = id  # Уникальный идентификатор заявки
        self.input_time = input_time  # Время поступления заявки
        self.start_service_time = None  # Время начала обслуживания
        self.served_in = []  # Каналы обслуживания, через которые прошла заявка
        self.rejected = False  # Флаг отказа в обслуживании
        self.output_time = None  # Время завершения обслуживания
        self.queue_time = 0  # Общее время ожидания в очереди
        self.service_time = 0  # Общее время обслуживания
        self.total_time_in_system = 0  # Время пребывания в системе

    def set_service_start(self, start_time, channel):
        """Фиксирует начало обслуживания заявки."""
        self.start_service_time = start_time
        self.served_in.append(channel)
        self.queue_time = start_time - self.input_time  # Время в очереди

    def set_service_end(self, end_time):
        """Фиксирует завершение обслуживания заявки."""
        self.output_time = end_time
        self.service_time = end_time - self.start_service_time
        self.total_time_in_system = end_time - self.input_time

    def mark_rejected(self):
        """Помечает заявку как отклоненную."""
        self.rejected = True
        self.total_time_in_system = 0  # Заявка не обслужена, значит, не находилась в системе
    def __str__(self):
        """Вывод информации о заявке."""
        status = "Отклонена" if self.rejected else "Обслужена"
        return (f"Заявка {self.id}\t: Время поступления {self.input_time}, "
                f"Статус: {status}, Время ожидания: {self.queue_time}, "
                f"Обслуживалась в: {self.served_in}, Время завершения: {self.output_time}")

In [5]:
# def service(requests,number_queue):
#     table={
#         "запросы": requests,
#         "канал 1": [],
#         "канал 2": [],
#         "обслуженно": [],
#         "отказ": []
#     }
#     for i in range(number_queue):
#         table[f"очередь {i+1}"] = []

#     choice_col=list(table.keys())[2:0:-1]
#     choice_queue=list(table.keys())[5:]
    
#     for req in requests:
#         is_suself=False
#         for kanal in choice_col:            
#             if not table[kanal] or table[kanal][-1][1]<=req:
#                 table[kanal].append((req,round(req+generator_gap(kanal_mu[kanal]),3)))
#                 table['обслуженно'].append((table[kanal][-1],table["запросы"].index(req)))
#                 is_suself=True
#                 break
            
#         if is_suself:
#             continue
        
#         near_time=min(table["канал 1"][-1][1],table["канал 2"][-1][1])
#         near_kanal="канал 2" if table["канал 2"][-1][1]==near_time else "канал 1"
        
#         if table[f"очередь {number_queue}"]:
#             if table[f"очередь {number_queue}"][-1][1]>=req:
#                 table['отказ'].append((req,table["запросы"].index(req)))
#                 is_suself=True
#                 continue
        
#         near_queue=number_queue
#         for i in range(number_queue-1,0,-1):
#             if not table[f"очередь {i}"] or table[f"очередь {i}"][-1][1]<=req:
#                 near_queue=i
              
#         enum=[near_kanal]+choice_queue[:near_queue]
#         enum.reverse()
#         prev=req
#         for i in range(1,len(enum)):
#             if not table[enum[i-1]] or table[enum[i-1]][-1][1]:
#                 if prev>table[enum[i]][-1][1]:
#                     print(req)
#                 a=(prev,table[enum[i]][-1][1])
#                 table[enum[i-1]].append(a)
#                 prev=table[enum[i]][-1][1]
      
#         table[near_kanal].append((prev,round(near_time+generator_gap(kanal_mu[near_kanal]),3)))
#         table['обслуженно'].append((table[near_kanal][-1],table["запросы"].index(req)))
                    
#     return table

# table=service(requests,3)
# for i in zip(table.keys(),table.values()):
#     print(len(i[1]),i)   

In [22]:
def service_class(requests, number_queue,kanal_mu):
    kanal_mu = kanal_mu  # Интенсивность обслуживанmия
    request_objects = [RequestCar(idx,requests[idx]) for idx in range(len(requests))]

    table = {
        "запросы": requests,
        "канал 1": [],
        "канал 2": [],
        "обслуженно": [],
        "отказ": [],
    }

    for i in range(number_queue):
        table[f"очередь {i+1}"] = []

    choice_col = list(table.keys())[2:0:-1]
    choice_queue = list(table.keys())[5:]

    for req_obj in request_objects:
        req = req_obj.input_time
        is_served = False

        for kanal in choice_col:
            if not table[kanal] or table[kanal][-1][1] <= req:
                start_time = req
                end_time = round(req + generator_gap(kanal_mu[kanal]), 3)
                table[kanal].append((start_time, end_time))
                table['обслуженно'].append((table[kanal][-1], req_obj.id))

                req_obj.set_service_start(start_time, kanal)
                req_obj.set_service_end(end_time)
                is_served = True
                break

        if is_served:
            continue

        near_time = min(table["канал 1"][-1][1], table["канал 2"][-1][1])
        near_kanal = "канал 2" if table["канал 2"][-1][1] == near_time else "канал 1"

        if number_queue==0 or table[f"очередь {number_queue}"]:
            if number_queue==0 or table[f"очередь {number_queue}"][-1][1] >= req:
                table['отказ'].append((req, req_obj.id))
                req_obj.mark_rejected()
                continue

        near_queue = number_queue
        for i in range(number_queue - 1, 0, -1):
            if not table[f"очередь {i}"] or table[f"очередь {i}"][-1][1] <= req:
                near_queue = i

        enum = [near_kanal] + choice_queue[:near_queue]
        enum.reverse()
        prev = req

        for i in range(1, len(enum)):
            if not table[enum[i - 1]] or table[enum[i - 1]][-1][1]:
                a = (prev, table[enum[i]][-1][1])
                table[enum[i - 1]].append(a)
                prev = table[enum[i]][-1][1]

        start_time = prev
        end_time = round(near_time + generator_gap(kanal_mu[near_kanal]), 3)
        table[near_kanal].append((start_time, end_time))
        table['обслуженно'].append((table[near_kanal][-1], req_obj.id))

        req_obj.set_service_start(start_time, near_kanal)
        req_obj.set_service_end(end_time)

    return table, request_objects
def filter_table_by_time_range(table, start_time, end_time):
    """
    Фильтрует данные в таблице, оставляя только те, которые попадают в указанный временной диапазон.
    
    Параметры:
        table (dict): Исходная таблица с данными
        start_time (float): Начальное время диапазона
        end_time (float): Конечное время диапазона
    
    Возвращает:
        dict: Отфильтрованная таблица
    """
    filtered_table = {}
    
    for key, data_list in table.items():
        filtered_data = []
        
        for item in data_list:
            # Определяем временные границы элемента
            if isinstance(item, (int, float)):
                # Для простых временных меток
                time = item
                time_start = time_end = time
            elif len(item) == 2:
                if isinstance(item[0], (int, float)):
                    # Для пар (время, ID)
                    time = item[0]
                    time_start = time_end = time
                else:
                    # Для интервалов (start, end)
                    time_start, time_end = item[0]
            else:
                continue  # Пропускаем неподдерживаемые форматы
            
            # Проверяем попадание в диапазон
            if  end_time >= time_end >= start_time and end_time >= time_start >= start_time:
                filtered_data.append(item)
        
        filtered_table[key] = filtered_data
    
    return filtered_table
def visualize_service_system(table, num_queues=3):
    fig, ax = plt.subplots(figsize=(100, 6))
    
    # Настройки внешнего вида
    colors = {
        'запросы': 'lightgray',
        'канал 1': 'lightblue',
        'канал 2': 'lightgreen',
        'очередь 1': 'mistyrose',
        'очередь 2': 'peachpuff',
        'очередь 3': 'lavender',
        'отказ': 'red',
        'обслуженно': 'limegreen'  # Новый цвет для обслуженных заявок
    }
    
    # Определение вертикальных позиций для каждой строки
    y_positions = {
        'запросы': 7,  # Сдвигаем вверх на 1
        'обслуженно': 6,  # Новая строка для обслуженных заявок
        'канал 1': 5,
        'канал 2': 4,
        'очередь 1': 3,
        'очередь 2': 2,
        'очередь 3': 1,
        'отказ': 0
    }
    
    # Отрисовка заявок (временные точки с индексами)
    for idx, time in enumerate(table['запросы']):
        # Точка заявки
        ax.plot(time, y_positions['запросы'], 'o', color=colors['запросы'], markersize=6)
        
        # Текст с индексом и временем
        ax.text(time, y_positions['запросы'] + 0.2, f"{idx}\n{time:.2f}", 
                ha='center', va='bottom', fontsize=7)
        
        # Вертикальная пунктирная линия через всю высоту графика
        ax.axvline(x=time, color='gray', linestyle=':', alpha=0.6, linewidth=0.8)
    
    # Отрисовка обслуженных заявок
    for segment, req_id in table['обслуженно']:
        start, end = segment
        # Прямоугольник для периода обслуживания
        ax.add_patch(patches.Rectangle(
            (start, y_positions['обслуженно'] - 0.4), end-start, 0.8,
            facecolor=colors['обслуженно'], edgecolor='black', alpha=0.6
        ))
        # Текст с ID заявки в центре отрезка
        ax.text((start + end)/2, y_positions['обслуженно'], 
                f"{req_id}", ha='center', va='center', fontsize=8)
    
    # Функция для отображения временных меток на отрезках
    def draw_segment_labels(start, end, y_pos, color='black'):
        duration = end - start
        mid_x = (start + end) / 2
        
        # Время начала
        ax.text(start, y_pos - 0.3, f"{start:.2f}", 
                ha='left', va='top', fontsize=7, color=color)
        
        # Время конца
        ax.text(end, y_pos - 0.3, f"{end:.2f}", 
                ha='right', va='top', fontsize=7, color=color)
        
        # Длительность в центре
        ax.text(mid_x, y_pos, f"{duration:.2f}", 
                ha='center', va='center', fontsize=8, color=color, weight='bold')
    
    # Отрисовка каналов обслуживания с подписями
    for channel in ['канал 1', 'канал 2']:
        for start, end in table[channel]:
            ax.add_patch(patches.Rectangle(
                (start, y_positions[channel] - 0.4), end-start, 0.8,
                facecolor=colors[channel], edgecolor='black'
            ))
            draw_segment_labels(start, end, y_positions[channel])
    
    # Отрисовка очередей с подписями
    for queue in [f'очередь {i+1}' for i in range(num_queues)]:
        if queue in table:
            for start, end in table[queue]:
                ax.add_patch(patches.Rectangle(
                    (start, y_positions[queue] - 0.4), end-start, 0.8,
                    facecolor=colors[queue], edgecolor='black'
                ))
                draw_segment_labels(start, end, y_positions[queue])
    
    # Отрисовка отказов
    for time, req_id in table['отказ']:
        ax.plot(time, y_positions['отказ'], 'ro', markersize=6)
        ax.text(time, y_positions['отказ'] - 0.3, f"{req_id}\n{time:.2f}", 
                ha='center', va='top', fontsize=7, color='red')
    
    # Настройка осей и подписей
    ax.set_yticks([y_positions[k] for k in y_positions])
    ax.set_yticklabels(['Заявки', 'Обслуженно', '1 канал', '2 канал', 
                       '1 место', '2 место', '3 место', 'Отказ'])
    ax.set_xlabel('Время (Тн)')
    ax.set_title('Схема обслуживания заявок с временными метками')
    
    # Настройка сетки
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.grid(which='major', linestyle='-', alpha=0.8)
    plt.minorticks_on()
    plt.tight_layout()
    
    plt.show()


In [7]:

def print_table(table, request_objects=None):
    for i in zip(table.keys(),table.values()):
        print(len(i[1]),i)
    if not request_objects:
        return
    for req_obj in request_objects:
        print(req_obj)


Отфильтрованно

In [8]:
def merge_intervals(intervals):
    """Объединяет пересекающиеся интервалы в один список."""
    if not intervals:
        return []
    intervals.sort()
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:  # Перекрытие
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged

def get_overlap(intervals1, intervals2):
    """Находит пересечение интервалов двух списков."""
    overlap = []
    i, j = 0, 0
    while i < len(intervals1) and j < len(intervals2):
        start1, end1 = intervals1[i]
        start2, end2 = intervals2[j]
        if end1 <= start2:
            i += 1
        elif end2 <= start1:
            j += 1
        else:  # Есть пересечение
            overlap.append((max(start1, start2), min(end1, end2)))
            if end1 < end2:
                i += 1
            else:
                j += 1
    return overlap
def get_free_intervals(busy_intervals, total_time):
    """Вычисляет интервалы простоя по интервалам занятости."""
    if not busy_intervals:
        return [(0, total_time)]
    
    free_intervals = []
    prev_end = 0
    
    for start, end in sorted(busy_intervals):
        if start > prev_end:
            free_intervals.append((prev_end, start))
        prev_end = max(prev_end, end)
    
    if prev_end < total_time:
        free_intervals.append((prev_end, total_time))
    
    return free_intervals

In [9]:
def probability_service1(table):
    probability=round(len(table['обслуженно'])/len(table['запросы'])*100,3)
    print(f"Вероятность обслуживания: {probability} %")
    return probability
def system_throughput2(table,time_N):
    throughput=round(len(table['обслуженно'])/time_N,3)
    print(f"Пропускная способность системы: {throughput} [шт/час]")
    return throughput
def probability_failure3(table):
    probability=round(len(table['отказ'])/len(table['запросы'])*100,3)
    print(f"Вероятность отказа: {probability} %")
    return probability
def single_channel_occupancy45(table, time_N):
    """
    Вычисляет вероятность занятости только одного канала в заданный период.

    :param channel1_intervals: список кортежей (start, end) для 1-го канала
    :param channel2_intervals: список кортежей (start, end) для 2-го канала
    :param time_N: общее время наблюдения
    :return: вероятность занятости только одного канала
    """
    # Объединяем интервалы занятости
    merged_channel1 = merge_intervals(table1['канал 1'])
    merged_channel2 = merge_intervals(table1['канал 2'])

    # Вычисляем общую занятость каждого канала
    total_channel1 = sum(end - start for start, end in merged_channel1)
    total_channel2 = sum(end - start for start, end in merged_channel2)

    # Вычисляем пересечение (время, когда оба канала заняты одновременно)
    overlap_intervals = get_overlap(merged_channel1, merged_channel2)
    total_overlap = sum(end - start for start, end in overlap_intervals)

    single_channel_time = (total_channel1 + total_channel2 - 2 * total_overlap) / time_N
    print(f"Вероятность занятости одного канала: {single_channel_time*100} %")
    
    second_channel_time=total_overlap / time_N
    print(f"Вероятность занятости двух канала: {second_channel_time*100} %")
    
    return single_channel_time,second_channel_time 
def average_occupied_channels6(single_channel_occupancy45):
    buf=1*single_channel_occupancy45[0]+2*single_channel_occupancy45[1]
    print(f"Среднее количество занятых каналов: {buf} канала\tПоказатель загрузки: {buf/2}%")
    return buf
def calculate_idle_probabilities789(table, time_N):
    """
    Вычисляет вероятность простоя хотя бы одного канала, двух каналов одновременно и всей системы.
    
    :param table: словарь с данными системы (должен содержать 'канал 1' и 'канал 2')
    :param time_N: общее время наблюдения
    :return: (P1*, P2*, Pc*) - вероятности простоя
    """
    # Объединяем и сортируем интервалы занятости для каждого канала
    merged_ch1 = merge_intervals(table['канал 1'])
    merged_ch2 = merge_intervals(table['канал 2'])
    
    # 1. Время простоя хотя бы одного канала (когда один свободен, а второй может быть занят)
    # Это сумма времени простоя каждого канала минус время простоя обоих одновременно
    
    # Время работы каждого канала
    busy_ch1 = sum(end - start for start, end in merged_ch1)
    busy_ch2 = sum(end - start for start, end in merged_ch2)
    
    # Время простоя каждого канала
    idle_ch1 = time_N - busy_ch1
    idle_ch2 = time_N - busy_ch2
    
    # 2. Время простоя обоих каналов одновременно (система полностью свободна)
    # Находим интервалы, когда оба канала свободны
    free_ch1 = get_free_intervals(merged_ch1, time_N)
    free_ch2 = get_free_intervals(merged_ch2, time_N)
    idle_both_intervals = get_overlap(free_ch1, free_ch2)
    idle_both = sum(end - start for start, end in idle_both_intervals)
    
    # 3. Время простоя хотя бы одного канала
    # Это когда либо первый свободен, либо второй, либо оба
    # Можно вычислить как: idle_ch1 + idle_ch2 - idle_both
    idle_at_least_one = idle_ch1 + idle_ch2 - idle_both
    
    # Вероятности
    P1 = idle_at_least_one / time_N  # Хотя бы один канал свободен
    P2 = idle_both / time_N          # Оба канала свободны (простой системы)
    Pc = P2                          # Для системы с двумя каналами Pc = P2
    
    print(f"Вероятность простоя хотя бы одного канала: {round(P1*100, 3)} %")
    print(f"Вероятность простоя двух каналов одновременно: {round(P2*100, 3)} %")
    print(f"Вероятность простоя всей системы: {round(Pc*100, 3)} %")
    
    return P1, P2, Pc

def calculate_queue_probabilities(queue_intervals_list, time_N):
    """
    Вычисляет вероятности для различных состояний очереди (от 0 до N заявок).
    
    :param queue_intervals_list: список списков кортежей (start, end), где каждый список - это заявки в очереди.
    :param time_N: общее время наблюдения.
    :return: словарь {P0з, P1з, ..., Pnз} - вероятности различных состояний очереди.
    """
    from itertools import combinations, chain

    
    
    num_places = len(queue_intervals_list)  # Количество мест в очереди

    # Объединяем интервалы для каждой заявки в очереди
    merged_intervals = [merge_intervals(intervals) for intervals in queue_intervals_list]

    # Вычисляем вероятности для каждого количества занятых мест в очереди
    state_durations = {i: 0 for i in range(num_places + 1)}

    # Анализируем моменты времени, когда занято i мест в очереди
    for i in range(1, num_places + 1):
        total_overlap_time = 0
        for combo in combinations(merged_intervals, i):
            overlap_intervals = combo[0]
            for other_intervals in combo[1:]:
                overlap_intervals = get_overlap(overlap_intervals, other_intervals)

            total_overlap_time += sum(end - start for start, end in overlap_intervals)

        # Коррекция времени, если превышает time_N
        state_durations[i] = min(total_overlap_time, time_N)

    # Вычисляем вероятность, что очередь пуста (P0з)
    total_busy_time = sum(state_durations[i] for i in range(1, num_places + 1))
    state_durations[0] = max(0, time_N - total_busy_time)  # Простои системы

    # Коррекция суммарного времени, если оно превысило время наблюдения
    total_time_used = sum(state_durations.values())
    if total_time_used > time_N:
        scale_factor = time_N / total_time_used
        state_durations = {i: duration * scale_factor for i, duration in state_durations.items()}

    # Нормируем вероятности
    probabilities = {f'P{i}з': state_durations[i] / time_N for i in range(num_places + 1)}

    # Выводим результат с правильным склонением
    def correct_word(n):
        if n == 1:
            return "заявка"
        elif 2 <= n <= 4:
            return "заявки"
        else:
            return "заявок"

    for i, p in enumerate(probabilities.values()):
        if i==0: continue
        print(f"Вероятность того, что в очереди будет {i} {correct_word(i)}: {round(p*100, 3)}%")

    return probabilities

def average_number_car_queue10(probabilities):
    i=0
    buf=0.0
    for p in probabilities.values():
        buf+=p*i
        i+=1
    
    print(f"Среднее количество заявок в очереди: {round(buf,3)} [шт]")
    return buf

def average_waiting_time13(queue_intervals_list, total_requests):
    from itertools import chain
    # Собираем все интервалы из всех мест в очереди
    all_intervals = list(chain.from_iterable(queue_intervals_list))
    merged_intervals = merge_intervals(all_intervals)
    total_waiting_time = sum(end - start for start, end in merged_intervals)
    # print(total_waiting_time)
    buf= total_waiting_time / total_requests if total_requests > 0 else 0
    print(f"Среднее время ожидания заявки в очереди: {round(buf,3)} часа ({round(buf,3)*60})")
    
    return buf
def average_service_time14(channel_intervals_list, total_requests):
    from itertools import chain

    # Собираем все интервалы из всех мест в очереди
    all_intervals = list(chain.from_iterable(channel_intervals_list))
    merged_intervals = merge_intervals(all_intervals)
    total_waiting_time = sum(end - start for start, end in merged_intervals)
    # print(total_waiting_time)
    buf= total_waiting_time / total_requests if total_requests > 0 else 0
    print(f"Среднее время обслуживания заявки: {round(buf,3)} часа ({round(buf,3)*60})")
    
    return buf
def average_application_time15(waiting, service):
    buf=waiting+service
    print(f"Среднее время нахождения заявки в системе: {round(buf,3)} часа ({round(buf,3)*60})")
    
    return buf
def average_applications_in_system_interval16(table, interval_length=1/6):
    """
    Вычисляет среднее количество заявок в системе методом разбиения на интервалы.
    
    :param table: словарь с данными о системе
    :param interval_length: длина подынтервала в часах (по умолчанию 10 минут = 1/6 часа)
    :return: среднее количество заявок в системе
    """
    # Получаем общее время наблюдения

    max_time = end_visor
    min_time = start_visor
    total_time = time_N
    
    # Количество подынтервалов
    K = int(total_time / interval_length) + 1
    
    # Создаем список моментов времени для проверки
    check_points = [min_time + i * interval_length for i in range(K)]
    check_points.append(max_time)  # Добавляем конечную точку
    
    # Функция для определения количества заявок в системе в момент времени t
    def apps_in_system_at_time(t):
        count = 0
        
        # Заявки в очередях
        for queue in ['очередь 1', 'очередь 2', 'очередь 3']:
            if queue in table:
                for start, end in table[queue]:
                    if start <= t < end:
                        count += 1
        
        # Заявки в каналах обслуживания
        for channel in ['канал 1', 'канал 2']:
            for start, end in table[channel]:
                if start <= t < end:
                    count += 1
        
        return count
    
    total_apps = 0
    prev_time = check_points[0]
    prev_count = apps_in_system_at_time(prev_time)
    
    for current_time in check_points[1:]:
        interval_length = current_time - prev_time
        total_apps += prev_count * interval_length
        prev_time = current_time
        prev_count = apps_in_system_at_time(current_time)
    
    average = total_apps / total_time if total_time > 0 else 0
    
    print(f"Среднее количество заявок в системе (метод интервалов): {round(average, 3)} [шт]\tИспользовано {K} интервалов по {round(interval_length*60,1)} минут")
    return average

Тестовый метод

In [ ]:
def get_table_params(table1,time_N):
    params=[]
    params.append(probability_service1(table1))
    params.append(system_throughput2(table1,time_N))
    params.append(probability_failure3(table1))
    sco=single_channel_occupancy45(table1,time_N)
    params.append(sco)
    params.append(average_occupied_channels6(sco))
    params.append(calculate_idle_probabilities789(table1,time_N))
    cqp=calculate_queue_probabilities([values for values in list(table1.values())[5:]],time_N)
    params.append(cqp)
    params.append(average_number_car_queue10(cqp))


    w=average_waiting_time13([values for values in list(table1.values())[5:]],len(table1['запросы']))
    s=average_service_time14([values for values in list(table1.values())[1:3]],len(table1['запросы']))
    params.append(w)
    params.append(s)
    params.append(average_application_time15(w,s))
    params.append(average_applications_in_system_interval16(table1))
    return params
    



def run(n_queue):
    requests=[round(generator_gap(lamd),3)]
    for i in range(0,N,1):
        requests.append(round(requests[-1]+generator_gap(lamd),3))

    table, request_objects = service_class(requests, n_queue ,kanal_mu)
    print_table(table)
    print()
    table1=filter_table_by_time_range(table,table["запросы"][-1]*0.05,table["запросы"][-1]*0.95)
    print_table(table1)

    start_visor=table1['обслуженно'][0][0][0]
    end_visor=table1['обслуженно'][-1][0][1]
    time_N=end_visor-start_visor
    print(f"Время наблюдения с {start_visor} по {end_visor} равно {time_N}")

    p=get_table_params(table1,time_N)
    # visualize_service_system(table)
    # visualize_service_system(table1)
    data={
        "Начальные данные": table,
        "Данные для исследования":table1,
        "Время наблюдения": time_N,
        "Начало наблюдения":start_visor,
        "Конец наблюдения": end_visor,
        "Контрольные значения таблицы": p
    }
    return data
table_data={}
for i in range(4):
    print(f"\nКоличество мест в очереди {i}")
    table_data[f"Количество мест в очереди {i}"]=run(i)

    


Количество мест в очереди 0
1001 ('запросы', [0.102, 0.13, 0.371, 0.523, 0.704, 0.724, 0.835, 0.894, 1.217, 1.247, 1.28, 1.337, 1.407, 1.462, 1.493, 1.794, 1.799, 1.833, 1.845, 1.964, 1.976, 2.044, 2.05, 2.131, 2.148, 2.546, 2.568, 2.941, 3.113, 3.128, 3.164, 3.201, 3.39, 3.392, 3.399, 3.456, 3.541, 3.671, 3.832, 3.85, 3.911, 4.107, 4.124, 4.143, 4.17, 4.459, 4.527, 4.609, 4.626, 4.678, 4.843, 5.034, 5.23, 5.406, 5.412, 5.442, 5.74, 5.761, 5.777, 5.828, 5.911, 5.936, 5.945, 6.204, 6.284, 6.311, 6.322, 6.644, 6.656, 6.704, 6.733, 6.768, 6.896, 6.948, 7.062, 7.238, 7.256, 7.262, 7.32, 7.444, 7.541, 7.671, 7.743, 8.013, 8.016, 8.158, 8.339, 8.361, 8.446, 8.547, 8.599, 8.715, 9.019, 9.111, 9.271, 9.325, 9.36, 9.414, 9.454, 9.692, 9.714, 9.787, 9.821, 9.929, 9.995, 10.272, 10.422, 10.681, 10.766, 10.865, 11.016, 11.41, 11.415, 11.425, 11.448, 11.465, 11.546, 11.789, 11.915, 11.94, 11.94, 11.977, 12.259, 12.266, 12.322, 12.406, 12.458, 12.903, 13.043, 13.375, 13.375, 13.522, 13.571, 13.588,

In [45]:
for i in table_data:
    print(i)
    for j in table_data[i]:
        print(j)
        try:
            print_table(table_data[i][j])
        except:
            print(table_data[i][j])
    print("\n\n")

Количество мест в очереди 0
Начальные данные
1001 ('запросы', [0.102, 0.13, 0.371, 0.523, 0.704, 0.724, 0.835, 0.894, 1.217, 1.247, 1.28, 1.337, 1.407, 1.462, 1.493, 1.794, 1.799, 1.833, 1.845, 1.964, 1.976, 2.044, 2.05, 2.131, 2.148, 2.546, 2.568, 2.941, 3.113, 3.128, 3.164, 3.201, 3.39, 3.392, 3.399, 3.456, 3.541, 3.671, 3.832, 3.85, 3.911, 4.107, 4.124, 4.143, 4.17, 4.459, 4.527, 4.609, 4.626, 4.678, 4.843, 5.034, 5.23, 5.406, 5.412, 5.442, 5.74, 5.761, 5.777, 5.828, 5.911, 5.936, 5.945, 6.204, 6.284, 6.311, 6.322, 6.644, 6.656, 6.704, 6.733, 6.768, 6.896, 6.948, 7.062, 7.238, 7.256, 7.262, 7.32, 7.444, 7.541, 7.671, 7.743, 8.013, 8.016, 8.158, 8.339, 8.361, 8.446, 8.547, 8.599, 8.715, 9.019, 9.111, 9.271, 9.325, 9.36, 9.414, 9.454, 9.692, 9.714, 9.787, 9.821, 9.929, 9.995, 10.272, 10.422, 10.681, 10.766, 10.865, 11.016, 11.41, 11.415, 11.425, 11.448, 11.465, 11.546, 11.789, 11.915, 11.94, 11.94, 11.977, 12.259, 12.266, 12.322, 12.406, 12.458, 12.903, 13.043, 13.375, 13.375, 13.522,